# CE49X: Introduction to Computational Thinking and Data Science for Civil Engineers
## Week 11: Unsupervised Learning & Clustering

**Instructor:** Dr. Eyuphan Koc
**Department of Civil Engineering, Bogazici University**
**Semester:** Spring 2026

Based on *Python Data Science Handbook* by Jake VanderPlas
Chapter 5: Machine Learning (Section 5.11 - k-Means Clustering)

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

from scipy.cluster.hierarchy import linkage, dendrogram

%matplotlib inline

np.random.seed(49)
plt.rcParams['figure.dpi'] = 100


## Quick Recap: Where Are We?

For the last five weeks every model we built was **supervised**:

- **Linear & logistic regression** (Week 6) — fit a parametric curve to known labels.
- **Naive Bayes** (Week 7) — model the probability of each class.
- **Support Vector Machines** (Week 8) — find the widest gap between classes.
- **Decision Trees & Random Forests** (Week 9) — ask a sequence of yes/no questions.
- **Neural Networks** (Week 10) — let many small units learn the answer together.

All of these have one thing in common: **somebody had to label the training data**. A structural engineer graded those Nepal buildings 1, 2, or 3. A radiologist marked tumours. Every supervised model is, deep down, learning to imitate a human expert.

Today we drop the labels.

**Welcome to unsupervised learning** — the family of techniques that finds structure in data *without being told what the answer is*. This week we focus on its most useful and intuitive member: **clustering**.


## Table of Contents

1. [What Is Unsupervised Learning?](#1.-What-Is-Unsupervised-Learning?)
2. [K-Means: The Workhorse](#2.-K-Means:-The-Workhorse)
3. [Picking k: The Elbow & Silhouette](#3.-Picking-k:-The-Elbow-&-Silhouette)
4. [Hierarchical Clustering & the Dendrogram](#4.-Hierarchical-Clustering-&-the-Dendrogram)
5. [DBSCAN: Density-Based Clustering](#5.-DBSCAN:-Density-Based-Clustering)
6. [Application: Clustering Global Earthquakes](#6.-Application:-Clustering-Global-Earthquakes)
7. [Summary & Practice](#7.-Summary-&-Practice)

---


## 1. What Is Unsupervised Learning?

Imagine you walk into a warehouse stacked with 50,000 unsorted concrete cylinders. You have no test reports, no project records, no labels at all — just the cylinders. You suspect they came from a handful of different mix designs, but nobody recorded which is which.

**Supervised learning cannot help you here.** It needs labels to learn from. But you can still ask a useful question:

> "Can I group these cylinders into a small number of *families* of similar specimens, so I can test one representative from each family instead of all 50,000?"

That is **clustering**. The computer never learns the *name* of a mix design — it just figures out that some specimens look like each other and some don't.


> **Key Insight: Supervised vs Unsupervised**
>
> | Aspect | Supervised | Unsupervised |
> |---|---|---|
> | Training data | features **+ labels** | features **only** |
> | Goal | predict the label for new data | discover structure in the data |
> | "Right" answer? | yes, on the test set | no — we judge by usefulness |
> | Example | "Is this building damage grade 1, 2, or 3?" | "Are there hidden groups of buildings?" |


> **Example: Civil Engineering Uses of Clustering**
>
> - **Building stock characterisation:** group a city's buildings into typological families (material × era × height) to plan retrofit programmes.
> - **Sensor health monitoring:** group bridge accelerometer readings into "normal operating regimes" and flag anything that doesn't fit a regime as anomalous.
> - **Geotechnical site investigation:** cluster soil samples by grain size + moisture + density to identify distinct soil layers without a geologist hand-labelling each one.
> - **Traffic pattern discovery:** group intersections by their daily flow profiles to design coordinated signal timing.


**Can you see the clusters?** Let's start with the easiest possible case — synthetic 2-D data where the answer is visually obvious. Your eye will solve it instantly; the rest of this lecture is about teaching a computer to do the same.


In [ ]:
# [LIVE] Generate a synthetic dataset with 4 well-separated clusters.
X_blobs, y_true = make_blobs(
    n_samples=400, centers=4, cluster_std=0.7, random_state=49
)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_blobs[:, 0], X_blobs[:, 1], s=30, color='steelblue',
           edgecolor='white', alpha=0.85)
ax.set_title('Synthetic 2-D data: how many clusters do you see?')
ax.set_xlabel('feature 1'); ax.set_ylabel('feature 2')
ax.grid(True, alpha=0.3)
plt.show()


You almost certainly answered "**four**", without even thinking about it. The hard part is not seeing clusters — it's *defining* what a cluster is precisely enough that a computer can find them too, and at scales (millions of points, dozens of dimensions) where your eye can no longer help.

**What this week does and does not cover.** Clustering is one corner of unsupervised learning. Two other large families exist:

- **Dimensionality reduction** (PCA, t-SNE, UMAP) — squashes high-dimensional data into 2-D or 3-D so you can plot it.
- **Anomaly detection** — finds the *odd-one-out* point that doesn't belong to any group.

Both are worth knowing, both deserve their own weeks, and both can be revisited if you want to extend a final project.

---


## 2. K-Means: The Workhorse

K-Means is the first clustering algorithm anyone learns, for three reasons:

1. The idea fits in one sentence.
2. The math fits on a napkin.
3. It is shockingly effective on well-behaved data.

**The one-sentence description**

> Pick *k* centre points; repeatedly (a) assign every data point to its nearest centre, and (b) move each centre to the mean of its assigned points — until nothing changes.

That's the whole algorithm. The reason it gets a whole section in every machine-learning course is that "until nothing changes" is doing a lot of work, and the choice of *k* is doing even more.


### 2.1 The objective: within-cluster sum of squares

K-Means is secretly minimising a single quantity — the **within-cluster sum of squares**, often abbreviated **WCSS** or just **inertia**:

$$
\text{WCSS} = \sum_{k=1}^{K} \sum_{\mathbf{x}_i \in C_k} \big\| \mathbf{x}_i - \boldsymbol{\mu}_k \big\|^2
$$

In words: for every point, take its Euclidean distance to its own cluster's centre, square it, and add them all up. The algorithm jiggles the centres to make that sum as small as possible.

> **Aside: Why Euclidean distance?**
> Because K-Means is built around the *mean* (whose name is right there in "K-**Means**"), and the mean is the point that minimises squared Euclidean distance. Switch the distance metric and the "mean" stops being optimal — which is why algorithms like *k-medoids* and *DBSCAN* exist. Practical consequence: **K-Means assumes all features live on comparable scales.** A feature ranging 0-10,000 will completely dominate a feature ranging 0-1. We will fix this with `StandardScaler` shortly.


### 2.2 [TOGETHER] Hand-worked K-Means on 7 points

The fastest way to demystify K-Means is to run it on a dataset small enough to follow on paper. Here are 7 points in 2-D. We will start with two deliberately bad initial centroids and watch them migrate to sensible positions.


In [ ]:
# [TOGETHER] Hand-worked K-Means: 7 points, k = 2, with iteration plots.
points = np.array([
    [1.0, 1.0],
    [1.5, 1.5],
    [2.0, 1.0],
    [3.5, 4.0],   # ambiguous middle point - will switch clusters!
    [5.0, 5.0],
    [5.5, 4.5],
    [6.0, 5.0],
])

# Start with two deliberately bad centroids.
centroids = np.array([[2.0, 4.0], [5.0, 2.0]])
colors = ['steelblue', 'indianred']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for it, ax in enumerate(axes):
    # ---- Assignment step ----
    d = np.linalg.norm(points[:, None, :] - centroids[None, :, :], axis=2)
    labels = d.argmin(axis=1)

    # ---- Plot current state ----
    for k in [0, 1]:
        ax.scatter(points[labels == k, 0], points[labels == k, 1],
                   s=180, color=colors[k], edgecolor='black', zorder=3,
                   label=f'cluster {k}' if it == 0 else None)
    ax.scatter(centroids[:, 0], centroids[:, 1], marker='X', s=350,
               color='black', zorder=4, label='centroid' if it == 0 else None)
    ax.set_title(f'Iteration {it}')
    ax.set_xlim(-1, 8); ax.set_ylim(-1, 7)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

    # ---- Print human-readable trace ----
    print(f'iter {it}: assignments = {labels.tolist()}')
    print(f'         centroids   = {np.round(centroids, 2).tolist()}')

    # ---- Update step (for next iteration) ----
    new_centroids = np.array([points[labels == k].mean(axis=0) for k in [0, 1]])
    centroids = new_centroids

axes[0].legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()


Read the printed trace alongside the figure. Notice in **Iteration 0** the middle point (3.5, 4.0) is closer to the (2, 4) centroid, so it joins the blue cluster. After we move the centroids to their cluster means, the centres slide apart — and on **Iteration 1** that same middle point is now closer to the new red centroid and switches sides. By **Iteration 2** the centroids settle and no point changes cluster, so the algorithm halts.

> **Key Insight: K-Means is just iterated averaging.**
>
> Each pass is one *assignment* step (Euclidean nearest neighbour) and one *update* step (take the mean). Repeating this two-step dance is guaranteed to never *increase* the WCSS — so it terminates, usually in a handful of iterations.


### 2.3 K-Means in scikit-learn

Now that we know what's happening under the hood, the sklearn version is a five-liner. Let's run it on the well-behaved 4-blob dataset from §1.


In [ ]:
# Fit K-Means with k = 4 on the synthetic blob data.
km = KMeans(n_clusters=4, n_init=10, random_state=49)
km.fit(X_blobs)

labels = km.labels_
centers = km.cluster_centers_

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels,
                     cmap='viridis', s=30, edgecolor='white', alpha=0.85)
ax.scatter(centers[:, 0], centers[:, 1], marker='X', s=250,
           color='red', edgecolor='black', linewidth=1.5, zorder=4,
           label='cluster centroid')
ax.set_title(f'K-Means with k=4 (inertia = {km.inertia_:.1f})')
ax.set_xlabel('feature 1'); ax.set_ylabel('feature 2')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


**A note on `n_init=10`.** K-Means is sensitive to its starting centroids — different initial random seeds can converge to different local optima. Sklearn re-runs the algorithm from `n_init` different random starts and keeps the best (lowest-WCSS) solution. The default `n_init=10` is almost always enough; you only need more if your data is huge and the answer matters a lot.

The smarter-than-random `k-means++` initialiser (sklearn's default) further reduces the risk of a bad start by spreading the initial centroids out across the data. That is why textbook horror stories about "K-Means converging to a terrible answer" are mostly historical — *if* you use the defaults.


### 2.4 Where K-Means fails: non-convex shapes

K-Means measures distance to a *single point* (the centroid), so it can only carve up space into **convex, blobby regions**. If your true clusters are crescent moons, rings, or spirals, K-Means cannot describe them — it will just slice them in half. Here is the canonical failure mode.


In [ ]:
# [LIVE] K-Means fails on two interleaving crescents.
X_moons, _ = make_moons(n_samples=300, noise=0.06, random_state=49)
km_moons = KMeans(n_clusters=2, n_init=10, random_state=49).fit(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], s=25, color='steelblue',
                edgecolor='white')
axes[0].set_title('Truth: two crescents')

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=km_moons.labels_,
                cmap='viridis', s=25, edgecolor='white')
axes[1].scatter(km_moons.cluster_centers_[:, 0], km_moons.cluster_centers_[:, 1],
                marker='X', s=250, color='red', edgecolor='black')
axes[1].set_title('K-Means: slices straight through both moons')

for ax in axes:
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


K-Means did exactly what it was told to do — find two centroids that minimise squared distance — and the result is geometrically correct but semantically wrong. We will fix this in §5 with **DBSCAN**, which doesn't think in centroids at all.

### 2.5 Pitfalls summary

Before moving on, hold these four warnings in your head whenever you reach for K-Means:

1. **You must pick *k*.** §3 is entirely about how.
2. **Features must be on comparable scales.** Always pipe through `StandardScaler`.
3. **It assumes convex, roughly equal-size, roughly spherical clusters.** Moons and rings will defeat it.
4. **It is sensitive to outliers.** A single point far from everything else will pull a centroid towards it. (Use `KMeans` on *clean* data, or switch to a robust variant like *k-medoids*.)


### 2.6 Why scaling matters: a quick demo


In [ ]:
# Same data, two features on wildly different scales.
rng = np.random.default_rng(49)
n_per = 100
centers_2d = np.array([[2, 2_000], [8, 8_000]])
X_unscaled = np.vstack([
    centers_2d[0] + rng.normal(scale=[1, 1000], size=(n_per, 2)),
    centers_2d[1] + rng.normal(scale=[1, 1000], size=(n_per, 2)),
])

# Fit K-Means on raw vs scaled features.
km_raw    = KMeans(n_clusters=2, n_init=10, random_state=49).fit(X_unscaled)
X_scaled  = StandardScaler().fit_transform(X_unscaled)
km_scaled = KMeans(n_clusters=2, n_init=10, random_state=49).fit(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(X_unscaled[:, 0], X_unscaled[:, 1], c=km_raw.labels_,
                cmap='viridis', s=30, edgecolor='white')
axes[0].set_title('Raw features\n(feature 2 dominates -> meaningless split)')
axes[0].set_xlabel('feature 1 (range 0-10)')
axes[0].set_ylabel('feature 2 (range 0-10,000)')

axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=km_scaled.labels_,
                cmap='viridis', s=30, edgecolor='white')
axes[1].set_title('After StandardScaler\n(both features contribute equally)')
axes[1].set_xlabel('feature 1 (standardised)')
axes[1].set_ylabel('feature 2 (standardised)')

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


On the raw data the y-axis is **a thousand times bigger** than the x-axis, so Euclidean distance is essentially y-distance. K-Means happily clusters by feature 2 alone and ignores feature 1. After `StandardScaler` both features have mean 0 and unit variance — and now both contribute. **This is not optional.** Forgetting to scale before clustering is the single most common bug in this whole topic.

---


## 3. Picking *k*: The Elbow & Silhouette

In every real problem the first question is the hardest: **how many clusters?** Unlike supervised learning, we cannot just hold out a test set and measure accuracy — there is no "true" answer to score against. So we have to fall back on heuristics.

Two heuristics dominate practice:

- **The elbow plot** — visual, fast, eye-of-the-beholder.
- **The silhouette score** — numeric, principled, slightly slower.

Use both. When they agree, trust the answer. When they disagree, that itself is information about your data.


### 3.1 The elbow method

The plan:

1. For *k* = 1, 2, 3, …, *k*<sub>max</sub>, fit K-Means and record the inertia (WCSS).
2. Plot inertia vs *k*.
3. Find the **bend** — the point where adding another cluster stops paying off much.

The intuition is that adding clusters always lowers the WCSS (in the extreme, every point is its own cluster and WCSS=0). The interesting *k* is where the curve transitions from "huge gain" to "marginal gain".


In [ ]:
# [LIVE] Elbow plot on the 4-blob synthetic data. The true k is 4.
ks = range(1, 11)
inertias = [
    KMeans(n_clusters=k, n_init=10, random_state=49).fit(X_blobs).inertia_
    for k in ks
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(ks), inertias, marker='o', color='steelblue', linewidth=2)
ax.axvline(4, color='indianred', linestyle='--', alpha=0.7, label='visible bend at k=4')
ax.set_xlabel('k (number of clusters)')
ax.set_ylabel('WCSS / inertia (lower = tighter clusters)')
ax.set_title('Elbow plot: find the bend')
ax.set_xticks(list(ks))
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


The curve drops steeply from *k*=1 to *k*=4, then flattens out: the elbow sits at *k* = 4, exactly matching the truth. On real data the bend is rarely this clean — sometimes you stare at a smooth curve and have to make a judgement call. That is when the silhouette score earns its keep.

### 3.2 The silhouette score

For each data point *i*, define:

- $a(i)$ = mean distance from point *i* to other points in **its own** cluster.
- $b(i)$ = mean distance from point *i* to points in the **nearest *other*** cluster.

Then the silhouette of point *i* is:

$$
s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}} \quad \in [-1, +1]
$$

Interpretation: $s(i) \approx 1$ means "I'm much closer to my own cluster than to any neighbouring cluster — well placed." $s(i) \approx 0$ means "I'm on the border." $s(i) < 0$ means "I'm *closer* to a neighbouring cluster than to my own — I'm in the wrong cluster."

The **average** silhouette across all points is one scalar that summarises how well-defined the clusters are. Higher is better.


In [ ]:
# Silhouette score swept over k = 2..8. (k=1 is undefined - there is no "other" cluster.)
ks = range(2, 9)
sil = [
    silhouette_score(X_blobs,
                     KMeans(n_clusters=k, n_init=10, random_state=49).fit_predict(X_blobs))
    for k in ks
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(ks), sil, marker='o', color='indianred', linewidth=2)
best_k = list(ks)[int(np.argmax(sil))]
ax.axvline(best_k, color='steelblue', linestyle='--', alpha=0.7,
           label=f'best k = {best_k}')
ax.set_xlabel('k')
ax.set_ylabel('mean silhouette (higher = better)')
ax.set_title('Silhouette sweep')
ax.set_xticks(list(ks))
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


The silhouette peaks at *k* = 4 — the elbow and the silhouette agree, confirming that 4 is the right number of clusters for this dataset.

> **Key Insight: Elbow vs silhouette in practice**
>
> The elbow is **heuristic and visual** — fast to compute but subject to taste. The silhouette is **numeric** — you can sort *k*'s and pick the maximum. Use the elbow for a first look, and the silhouette to break ties or argue with a colleague.

---


## 4. Hierarchical Clustering & the Dendrogram

K-Means asks for *k* up front. **Hierarchical clustering** refuses to commit. It builds a *tree* (called a **dendrogram**) showing how points and groups merge into ever-larger clusters as we relax the similarity requirement. You decide afterwards where to cut the tree.

There are two flavours:

- **Agglomerative** (bottom-up): start with each point in its own cluster; repeatedly merge the closest pair. This is by far the more common variant — and the one sklearn implements.
- **Divisive** (top-down): start with one big cluster; recursively split. Rarer and slower.

We'll use agglomerative.


### 4.1 Linkage criteria — how do we measure "closeness" between *clusters*?

We know how to measure the distance between two *points* (Euclidean). But once points are in groups, "distance between two clusters" needs defining. Four common choices:

- **Single linkage:** distance between the two *closest* points across clusters. Tends to produce long, stringy "chain" clusters.
- **Complete linkage:** distance between the two *farthest* points. Tends to produce compact, equal-size clusters.
- **Average linkage:** mean of all pairwise distances. A reasonable middle ground.
- **Ward linkage:** merge the pair that increases WCSS the least. Tends to produce compact spherical clusters — most similar in spirit to K-Means.

When in doubt, start with **Ward**.


In [ ]:
# [LIVE] Build a dendrogram on a small subset (so labels are readable).
X_small, _ = make_blobs(n_samples=20, centers=4, cluster_std=0.6, random_state=49)
Z = linkage(X_small, method='ward')

fig, ax = plt.subplots(figsize=(11, 5))
dendrogram(Z, ax=ax, color_threshold=4, above_threshold_color='gray')
ax.axhline(4, color='indianred', linestyle='--', alpha=0.7,
           label='cut here -> 4 clusters')
ax.set_xlabel('point index')
ax.set_ylabel('merge distance (Ward)')
ax.set_title('Dendrogram of 20 points (Ward linkage)')
ax.legend()
plt.show()


Read the dendrogram from the bottom up. Each leaf is one of the 20 original points. Two leaves joined by a horizontal bar were merged into a cluster at that height. Heights are large when the merged sub-clusters were dissimilar.

To pick *k*, draw a horizontal line: every vertical line it cuts is one cluster. Lowering the line gives more, finer clusters; raising it gives fewer, coarser clusters. The dashed line above cuts the tree into **4 clusters** — note that it crosses a large gap, which is a strong signal that 4 is a natural choice for this dataset.

> **Key Insight: A dendrogram lets you defer the choice of k.**
> You generate the tree once, then cut it at whatever granularity your problem demands. K-Means cannot do this — every new *k* needs a new fit.


In [ ]:
# Agglomerative clustering with Ward linkage, k = 4. Compare side-by-side with K-Means.
agg = AgglomerativeClustering(n_clusters=4, linkage='ward').fit(X_blobs)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=km.labels_,
                cmap='viridis', s=25, edgecolor='white')
axes[0].set_title('K-Means (k=4)')

axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=agg.labels_,
                cmap='viridis', s=25, edgecolor='white')
axes[1].set_title('Agglomerative, Ward linkage (k=4)')

for ax in axes:
    ax.set_xlabel('feature 1'); ax.set_ylabel('feature 2')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


On well-separated blobs the two algorithms produce essentially identical partitions (cluster *colours* may differ — they are arbitrary labels). The advantage of hierarchical clustering only emerges when the dataset *isn't* a clean blob:

- You want a **hierarchy of granularities** (e.g. soils → soil types → soil sub-types).
- You don't know *k* in advance and want to *see* the structure before committing.
- The clusters have **very different sizes** — K-Means fights this, Ward handles it gracefully.

**Costs.** Agglomerative clustering is $O(n^2)$ in memory and at least $O(n^2 \log n)$ in time. Above ~50,000 points it becomes painful. For very large datasets, prefer K-Means or MiniBatchKMeans.

---


## 5. DBSCAN: Density-Based Clustering

Both algorithms so far measure "closeness to a centre point". DBSCAN — **Density-Based Spatial Clustering of Applications with Noise** — throws that mental model out.

> **Definition: a cluster, the DBSCAN way.**
> A cluster is a **maximal set of points connected through dense neighbourhoods**. Points in low-density regions are not assigned to any cluster — they are **noise**.

This is a fundamentally different question. K-Means asks *"which centroid is closest?"*. DBSCAN asks *"is this point in a crowd?"*. Those two questions give different answers as soon as your clusters aren't blobs.

### 5.1 The two parameters

DBSCAN has two knobs:

- **`eps`** — the neighbourhood radius. "How close is close enough?"
- **`min_samples`** — the density threshold. "How many neighbours do I need to count as being in a crowd?"

And three kinds of point emerge:

- **Core point:** has at least `min_samples` neighbours within `eps`. Lives in a crowd.
- **Border point:** doesn't itself have `min_samples` neighbours, but is within `eps` of a core point.
- **Noise point:** neither. Floats alone.

Clusters are formed by chaining core points together through their neighbourhoods. Two core points in each other's `eps`-ball are in the same cluster; their border-point hangers-on come along for the ride.


In [ ]:
# [LIVE] DBSCAN succeeds on the moons where K-Means failed.
dbs_moons = DBSCAN(eps=0.20, min_samples=5).fit(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km_moons.labels_,
                cmap='viridis', s=25, edgecolor='white')
axes[0].set_title('K-Means: slices through both moons')

# DBSCAN convention: label -1 == noise. Plot those in gray.
mask = dbs_moons.labels_ == -1
axes[1].scatter(X_moons[~mask, 0], X_moons[~mask, 1],
                c=dbs_moons.labels_[~mask], cmap='viridis',
                s=25, edgecolor='white')
axes[1].scatter(X_moons[mask, 0], X_moons[mask, 1],
                color='lightgray', s=25, edgecolor='white', label='noise (-1)')
axes[1].set_title(f'DBSCAN: respects the geometry  '
                  f'({(dbs_moons.labels_ != -1).sum()} clustered, '
                  f'{mask.sum()} noise)')
axes[1].legend(loc='lower left')

for ax in axes:
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Same story on concentric circles.
X_circ, _ = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=49)
dbs_circ = DBSCAN(eps=0.15, min_samples=5).fit(X_circ)
km_circ  = KMeans(n_clusters=2, n_init=10, random_state=49).fit(X_circ)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(X_circ[:, 0], X_circ[:, 1], c=km_circ.labels_,
                cmap='viridis', s=25, edgecolor='white')
axes[0].set_title('K-Means on concentric rings (fails)')

mask = dbs_circ.labels_ == -1
axes[1].scatter(X_circ[~mask, 0], X_circ[~mask, 1],
                c=dbs_circ.labels_[~mask], cmap='viridis',
                s=25, edgecolor='white')
axes[1].scatter(X_circ[mask, 0], X_circ[mask, 1],
                color='lightgray', s=25, edgecolor='white', label='noise')
axes[1].set_title('DBSCAN on concentric rings (works)')
axes[1].legend(loc='lower left')

for ax in axes:
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 5.2 Picking `eps`: the k-distance plot

`eps` is the trickiest parameter to set. Too small → every point becomes its own noise speck. Too large → all points merge into a single mega-cluster. The standard heuristic is the **k-distance plot**:

1. For every point, find its distance to its *k*-th nearest neighbour (where *k* = `min_samples`).
2. Sort those distances and plot them.
3. Pick `eps` at the **knee** of the curve — the elbow-like bend where distances start shooting up.

The intuition: in dense regions the *k*-th neighbour is close; in sparse regions it's far. The knee marks the transition.


In [ ]:
# k-distance plot for the moons data (k = min_samples = 5).
k = 5
nn = NearestNeighbors(n_neighbors=k).fit(X_moons)
dists, _ = nn.kneighbors(X_moons)
kth_dist = np.sort(dists[:, -1])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(kth_dist, color='steelblue', linewidth=2)
ax.axhline(0.20, color='indianred', linestyle='--', alpha=0.7,
           label='chosen eps = 0.20')
ax.set_xlabel('points sorted by k-th-neighbour distance')
ax.set_ylabel(f'distance to the {k}-th nearest neighbour')
ax.set_title('k-distance plot - pick eps at the knee')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


The curve is flat until about `eps ≈ 0.2`, then accelerates upward — that's the knee, and our chosen `eps` lands right on it. If you grab `eps` *below* the knee you fragment the dense regions; *well above* the knee you melt everything into one giant cluster.

> **Key Insight: Two questions, two algorithms.**
> *K-Means:* "Which centroid is closest to me?"
> *DBSCAN:*  "Am I in a crowd? If yes, which one?"
> These are not interchangeable. Match the question to the geometry of your data.

### 5.3 When NOT to use DBSCAN

DBSCAN is brilliant for arbitrary-shaped clusters and for separating signal from noise — but it has its own failure modes:

- **Varying density.** If one cluster is dense and another is sparse, a single global `eps` cannot serve both. (Solution: try HDBSCAN, an adaptive cousin.)
- **High dimensions.** Distance loses meaning above ~10 dimensions (the curse of dimensionality bites). Either reduce dimensions first or pick a different algorithm.

---


## 6. Application: Clustering Global Earthquakes

It's time to drop the synthetic data and try something real. Here is our motivating question:

> **Given one year of every significant earthquake on Earth — and no information beyond *where* and *when* they happened — can unsupervised learning rediscover the major seismic zones?**

Geologists already know the answer: yes, earthquakes cluster along **plate boundaries**. The Pacific Ring of Fire is the most famous example, but there are many others — the Mid-Atlantic Ridge, the Mediterranean–Himalayan belt, and so on. The point of this exercise is not to *teach* us where earthquakes happen — it's to ask whether an algorithm with zero geological knowledge will discover the same structure on its own.

If yes, that is a powerful demonstration of what unsupervised learning is good for: **finding structure that experts already know is there**, on datasets where they *don't* yet know what the structure is.


### 6.1 The dataset

Source: the USGS FDSN earthquake catalog (`https://earthquake.usgs.gov/fdsnws/event/1/`), one calendar year (2024-01-01 to 2024-12-31), every event of magnitude 4.5 or larger. Roughly 6,000 events. See `data/README.md` for the full citation and a regenerator script.

Columns:

| Column | Description |
|---|---|
| `time` | Event time (ISO-8601 UTC) |
| `latitude`, `longitude` | Epicentre, decimal degrees |
| `depth_km` | Hypocentre depth below mean sea level |
| `magnitude` | Reported magnitude |
| `mag_type` | Magnitude scale (mb, mww, …) |
| `place` | Human-readable nearest location |


In [ ]:
df = pd.read_csv('data/usgs_earthquakes.csv')
print(f'{len(df):,} earthquakes')
print(f'date range: {df["time"].min()}  ->  {df["time"].max()}')
print(f'magnitude range: {df["magnitude"].min():.1f}  ->  {df["magnitude"].max():.1f}')
df.head()


In [ ]:
df.describe()[['latitude', 'longitude', 'depth_km', 'magnitude']]


### 6.2 Look at the data first

Before running a single algorithm: plot it. With longitude on the x-axis and latitude on the y-axis, an earthquake catalogue *is* a map of the world's seismic activity.


In [ ]:
# [LIVE] World scatter: each earthquake is one dot, scaled by magnitude.
fig, ax = plt.subplots(figsize=(13, 6))
sizes = 4 ** (df['magnitude'] - 4)   # gentle exponential scaling
ax.scatter(df['longitude'], df['latitude'], s=sizes,
           color='indianred', alpha=0.5, edgecolor='none')
ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
ax.set_xticks(range(-180, 181, 60)); ax.set_yticks(range(-90, 91, 30))
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
ax.set_title(f'{len(df):,} earthquakes, M >= 4.5, 2024')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.show()


The plate-boundary structure jumps out without any algorithm at all. You can see the Pacific Ring of Fire (the upside-down U around the Pacific), the Mid-Atlantic Ridge (the thin vertical band through the Atlantic), the Mediterranean–Himalayan belt, and several smaller belts. That's the structure clustering must rediscover.


### 6.3 Step 1 — K-Means on (latitude, longitude)

Latitude and longitude are already on the same scale (-90/+90 and -180/+180 degrees), so scaling is less critical here than usual. We will still standardise, partly out of habit and partly because it makes the next stage (with `depth` and `magnitude`) consistent.


In [ ]:
X_geo = df[['latitude', 'longitude']].to_numpy()
X_geo_scaled = StandardScaler().fit_transform(X_geo)

# Elbow plot for the earthquake data.
ks = range(1, 13)
inertias = [
    KMeans(n_clusters=k, n_init=10, random_state=49).fit(X_geo_scaled).inertia_
    for k in ks
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(ks), inertias, marker='o', color='steelblue', linewidth=2)
ax.set_xlabel('k'); ax.set_ylabel('inertia (WCSS)')
ax.set_title('Elbow plot: earthquakes, lat/lon')
ax.set_xticks(list(ks))
ax.grid(True, alpha=0.3)
plt.show()


The bend isn't razor-sharp — real data rarely is. It softens somewhere around *k* = 6 to 8. We'll pick **k = 7** as a reasonable compromise; you should try a couple of values in the practice exercises.


In [ ]:
# Fit K-Means with k = 7 and plot clusters on a world map.
k = 7
km_eq = KMeans(n_clusters=k, n_init=10, random_state=49).fit(X_geo_scaled)
df['kmeans_cluster'] = km_eq.labels_

fig, ax = plt.subplots(figsize=(13, 6))
ax.scatter(df['longitude'], df['latitude'], c=df['kmeans_cluster'],
           cmap='tab10', s=8, alpha=0.7, edgecolor='none')
ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
ax.set_xticks(range(-180, 181, 60)); ax.set_yticks(range(-90, 91, 30))
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
ax.set_title(f'K-Means (k = {k}) on earthquake (lat, lon)')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.show()


K-Means produces a tidy partition of the world into 7 geographic regions of seismicity. Roughly: a Pacific Rim cluster (probably split into northern, western and eastern halves), a South-American cluster, a Eurasian belt, a Mid-Atlantic strip, an Indonesian/Australian cluster, and so on — though the exact split depends on the random seed.

**But notice something uncomfortable.** Every dot got a colour, including the handful of isolated quakes floating in the middle of the Pacific, the deep South Atlantic, and so on. K-Means **cannot say "I don't know"**. It will always assign every point to *some* cluster, no matter how clearly that point is an outlier. That's where DBSCAN earns its place.


### 6.4 Step 2 — DBSCAN on (latitude, longitude)

DBSCAN refuses to label sparse points. They get the special label `-1` = *noise*. For a global earthquake map this is the right behaviour: a single mid-ocean event 5,000 km from anything else really *shouldn't* be lumped into a "Pacific Rim cluster" just to keep the bookkeeping tidy.


In [ ]:
# DBSCAN on standardised (lat, lon). eps is in standardised units;
# we pick it via a quick k-distance plot below.
k = 5
nn = NearestNeighbors(n_neighbors=k).fit(X_geo_scaled)
dists, _ = nn.kneighbors(X_geo_scaled)
kth_dist = np.sort(dists[:, -1])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(kth_dist, color='steelblue', linewidth=2)
ax.axhline(0.07, color='indianred', linestyle='--', alpha=0.7,
           label='chosen eps = 0.07')
ax.set_xlabel('points sorted by k-th-neighbour distance')
ax.set_ylabel(f'distance to {k}-th NN (standardised units)')
ax.set_title('k-distance plot, earthquake (lat, lon)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


In [ ]:
dbs_eq = DBSCAN(eps=0.07, min_samples=5).fit(X_geo_scaled)
df['dbscan_cluster'] = dbs_eq.labels_

n_clusters = (df['dbscan_cluster'].max() + 1) if df['dbscan_cluster'].max() >= 0 else 0
n_noise = (df['dbscan_cluster'] == -1).sum()
print(f'DBSCAN found {n_clusters} clusters and labelled {n_noise:,} '
      f'events ({100*n_noise/len(df):.1f}%) as noise.')

fig, ax = plt.subplots(figsize=(13, 6))
noise_mask = df['dbscan_cluster'] == -1
# Noise first, faint, in gray.
ax.scatter(df.loc[noise_mask, 'longitude'], df.loc[noise_mask, 'latitude'],
           color='lightgray', s=8, alpha=0.6, edgecolor='none', label='noise')
# Clustered points coloured.
ax.scatter(df.loc[~noise_mask, 'longitude'], df.loc[~noise_mask, 'latitude'],
           c=df.loc[~noise_mask, 'dbscan_cluster'], cmap='tab20',
           s=8, alpha=0.85, edgecolor='none')
ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
ax.set_xticks(range(-180, 181, 60)); ax.set_yticks(range(-90, 91, 30))
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
ax.set_title(f'DBSCAN on earthquake (lat, lon): '
             f'{n_clusters} clusters + {n_noise:,} noise points')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.show()


This is qualitatively different from the K-Means map. DBSCAN has carved seismic activity into many distinct **belts and arcs**, each one closely tracing a real plate boundary, while the isolated grey dots in the open ocean and continental interiors are correctly flagged as noise.

> **Key Insight: DBSCAN refused to label mid-ocean isolated quakes — and that refusal is itself a finding.**
> A clustering algorithm that confidently labels everything tells you nothing about which points are well-explained and which aren't. DBSCAN's noise label is a built-in *honesty mechanism*.

### 6.5 Side-by-side comparison

| | K-Means (k=7) | DBSCAN (eps≈0.07) |
|---|---|---|
| **Output** | Every point gets a cluster | Many clusters + noise |
| **Cluster shape** | Convex regions | Follows the geometry of plate boundaries |
| **What it answers** | "Which seismic region is this earthquake closest to?" | "Is this earthquake part of an active fault zone?" |
| **Best for** | Coarse regional summaries | Finding the boundaries themselves |


### 6.6 Extension — adding depth and magnitude

So far we clustered on geography alone. But each earthquake has two more physically meaningful features: **depth** (shallow crustal quakes vs deep subduction-zone events) and **magnitude**. Adding them turns the clustering question from "where do quakes happen?" to "what *kinds* of quake happen where?".


In [ ]:
# Cluster on 4 features: lat, lon, depth_km, magnitude. Scale first!
X4 = df[['latitude', 'longitude', 'depth_km', 'magnitude']].to_numpy()
X4_scaled = StandardScaler().fit_transform(X4)

dbs4 = DBSCAN(eps=0.30, min_samples=10).fit(X4_scaled)
df['dbscan4_cluster'] = dbs4.labels_

n_clusters4 = (df['dbscan4_cluster'].max() + 1) if df['dbscan4_cluster'].max() >= 0 else 0
n_noise4 = (df['dbscan4_cluster'] == -1).sum()
print(f'4-feature DBSCAN: {n_clusters4} clusters, {n_noise4:,} noise points')

# Summarise each cluster's centre in physical units.
non_noise = df[df['dbscan4_cluster'] >= 0]
summary = non_noise.groupby('dbscan4_cluster').agg(
    n=('magnitude', 'size'),
    lat=('latitude', 'mean'),
    lon=('longitude', 'mean'),
    depth_km=('depth_km', 'mean'),
    magnitude=('magnitude', 'mean'),
).round(2).sort_values('n', ascending=False)
summary.head(10)


Each row of that summary table represents one *type* of seismic event the algorithm discovered — distinguished not just by where it happens, but by how deep and how strong. Cross-referencing the deeper clusters with a tectonic map will show that they line up almost perfectly with **subduction zones**, where one plate dives beneath another and produces deep, large-magnitude events. Shallow, lower-magnitude clusters tend to fall on transform faults and ridges. The algorithm rediscovered tectonics on its own.

This is the payoff of unsupervised learning: take real-world data with no labels, run a clustering algorithm with sensible defaults, and find that the groups it produces correspond to physical reality. That correspondence is what tells you the structure is real — not an artefact of the algorithm.

---


## 7. Summary & Practice

### Key Takeaways

- **Unsupervised learning discovers structure without labels.** Its judgement is its own usefulness, not test-set accuracy.
- **K-Means** is fast, simple, and the right first choice — *if* you can pick *k* and your clusters are roughly spherical.
- **Hierarchical (agglomerative) clustering** produces a dendrogram that lets you defer the choice of *k* and inspect cluster structure at every scale.
- **DBSCAN** finds clusters of arbitrary shape and explicitly labels noise. Tune via the k-distance plot.
- **Always scale features first** — the only universal rule in this whole topic.
- **Use elbow + silhouette together** to pick *k* for K-Means.
- The choice of clustering algorithm encodes a definition of "what counts as a cluster". Pick the one whose definition matches your data's geometry.

### A quick decision flow

> Need to cluster a lot of data fast, you can pick *k*, clusters look blob-shaped? → **K-Means.**
> Want a hierarchy / unsure about *k* / fewer than ~50k points? → **Agglomerative (Ward).**
> Clusters are irregular shapes, or you want noise rejected? → **DBSCAN.**


## [PRACTICE] Practice Exercises

Try these on your own. Exercises 5 and 6 are excellent warm-ups for your final project.

1. **Exercise 1 (Warm-up — silhouette sweep on real data):** Compute the mean silhouette score for *k* = 2 … 10 on the standardised earthquake `(latitude, longitude)` data and plot the result. Which *k* does the silhouette pick? Is it the same as the elbow choice we used in §6.3?

2. **Exercise 2 (The eps dial):** Run DBSCAN on `X_geo_scaled` for `eps ∈ {0.03, 0.07, 0.15, 0.30}` with `min_samples=5`. For each run, report the number of clusters and the percentage of points labelled noise. Describe in one sentence what happens at the two extremes.

3. **Exercise 3 (Scaling matters):** Re-run the §6.3 K-Means on the **un**scaled `(latitude, longitude)` data, plot the clusters, and compare to the scaled version. Are the clusters meaningfully different? Why or why not? (Hint: think about the *ranges* of latitude and longitude.)

4. **Exercise 4 (Linkage shootout):** Generate `make_moons(n_samples=300, noise=0.05)`. Run `AgglomerativeClustering(n_clusters=2)` with `linkage ∈ {single, complete, average, ward}`. Plot all four results in one 2×2 figure. Which linkage handles the moons correctly? Write 1–2 sentences explaining *why* the others fail.

5. **Exercise 5 (Project warm-up — what does each cluster mean?):** Take the §6.6 4-feature DBSCAN result. For each cluster (excluding noise), produce:
   - mean depth in km,
   - mean magnitude,
   - 3 most common entries of the `place` column.

   Then assign each cluster a one-line physical interpretation (e.g. "deep South-American subduction zone", "shallow Mid-Atlantic ridge"). This is exactly the type of analysis you'll do on your final project.

6. **Exercise 6 (Challenge — your own data):** Pick *any* tabular dataset relevant to your final-project domain (sensor data, soils, structural inventory, traffic flows, building energy, hydrology, …). Apply:
   - K-Means (with an elbow plot to pick *k*), and
   - DBSCAN (with a k-distance plot to pick `eps`).

   Submit:
   - (a) one figure per algorithm,
   - (b) the silhouette score for the K-Means result,
   - (c) **one paragraph** of engineering interpretation: what does each cluster represent in the real world, and how would you actually use this grouping?


In [ ]:
# Your code here


---

### Questions?

**Dr. Eyuphan Koc**
eyuphan.koc@bogazici.edu.tr
